# Jinjalume reverse engineering and implementation backlog

This notebook documents the current behavior of the Jinjalume repository and turns the observed gaps into an implementation-ready backlog. It is intentionally evidence-led: items marked as missing or partial are derived from the source, tests, package metadata, and development configuration present in this checkout.

**Repository snapshot:** `v0.1.0` / `main` plus the current working tree

**Scope:** Flask integration, Jinja macro contracts, HTML/accessibility behavior, CSS packaging, tests, and project documentation.

The notebook does not change application behavior. Its executable cells provide a repeatable inventory and smoke check that can be rerun after each implementation slice.

## Public GitHub issue worklist

The current public issue list was inspected read-only. The five open issues map to the local implementation as follows:

| Issue | Request | Local status |
| --- | --- | --- |
| [#6](https://github.com/phcodesage/jinjalume/issues/6) | Native select component | Implemented with selected/disabled options, help/error states, docs, demo, and tests |
| [#3](https://github.com/phcodesage/jinjalume/issues/3) | Token-based dark mode | Implemented with semantic CSS variables, `data-theme`, docs, toggle, and snapshots |
| [#4](https://github.com/phcodesage/jinjalume/issues/4) | Copy-paste component docs | Implemented in `docs/components.md` and the full gallery |
| [#2](https://github.com/phcodesage/jinjalume/issues/2) | Optional HTMX example | Implemented at `/htmx` with a plain POST fallback and fragment response |
| [#1](https://github.com/phcodesage/jinjalume/issues/1) | Visual regression coverage | Implemented with Playwright desktop/mobile/dark snapshots and CI |

No GitHub comments, issue state changes, commits, or pushes are performed by this notebook or the local implementation workflow.

## Executive findings

- **Implemented:** `Jinjalume` registers package templates through Jinja's `PackageLoader` while preserving application templates.
- **Implemented:** ten plain Jinja macros cover buttons, badges, alerts, cards, inputs, selects, textareas, avatars, spinners, and native dialogs.
- **Intentional boundary:** Tailwind CSS is a development/demo asset. The Python package ships templates, not a compiled stylesheet.
- **No literal stubs:** there are no `TODO`, `FIXME`, `pass`, or `NotImplemented` markers in the tracked source. The missing work is therefore mostly contract hardening, accessibility behavior, extensibility, and tests.
- **Current verification:** 9 pytest tests pass; Ruff passes; the Tailwind build passes; wheel and source distribution builds pass; 6 Playwright visual tests pass locally.

The five current public issue requests are now represented locally. The remaining follow-up backlog is mostly API hardening: arbitrary HTML attributes, explicit form IDs, disabled link semantics, and broader contract coverage.

In [ ]:
import re
from pathlib import Path

# Locate the repository even when the notebook is launched from notebooks/.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "jinjalume").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the Jinjalume checkout.")

PACKAGE = ROOT / "jinjalume"
TEMPLATES = PACKAGE / "templates" / "jinjalume" / "components"
TESTS = ROOT / "tests"
print(f"Repository: {ROOT}")
print(f"Component templates: {len(list(TEMPLATES.glob('*.html')))}")
print(f"Tests: {len(list(TESTS.rglob('test_*.py')))} test module(s)")

In [ ]:
# Source inventory: macro names, signatures, and the files that define them.
macro_pattern = re.compile(r"{%\s*macro\s+(\w+)\((.*?)\)", re.S)
components = []
for path in sorted(TEMPLATES.glob("*.html")):
    source = path.read_text(encoding="utf-8")
    matches = macro_pattern.findall(source)
    components.append({
        "file": path.relative_to(ROOT).as_posix(),
        "macros": [name for name, _ in matches],
        "signatures": ["".join(signature.split()) for _, signature in matches],
    })

for component in components:
    print(component["file"])
    for name, signature in zip(component["macros"], component["signatures"]):
        print(f"  {name}({signature})")

## Reconstructed runtime flow

```text
Flask application
    │
    ├── Jinjalume(app)
    │       └── init_app(app)
    │              └── prepend PackageLoader("jinjalume", "templates")
    │
    ├── Jinja template imports a macro by package path
    │       └── jinjalume/components/<component>.html
    │
    └── macro emits server-rendered HTML
            └── application-owned Tailwind pipeline supplies CSS
```

### Confirmed design choices

1. The extension owns template discovery only; it does not replace the app's static folder or CSS configuration.
2. Application templates remain available because the package loader is composed with the existing loader.
3. Component APIs are macro calls, not Python view helpers. The caller passes text and a small set of styling/behavior options.
4. Interactivity is deliberately progressive: the modal uses native `<dialog>` and `method="dialog"`; no JavaScript bundle is shipped.

In [ ]:
from flask import Flask, render_template_string

from jinjalume import Jinjalume

# Runtime smoke check: validates the package loader and representative macro output.
app = Flask("jinjalume_reverse_engineering")
extension = Jinjalume(app)

template = """
{% from "jinjalume/components/button.html" import button %}
{% from "jinjalume/components/input.html" import input_field %}
{% from "jinjalume/components/modal.html" import modal %}
{{ button("Save", variant="primary", type="submit") }}
{{ input_field("email", label="Email", error="Required") }}
{% call modal("details", "Details", description="More information") %}Body{% endcall %}
"""

with app.app_context():
    rendered = render_template_string(template)

assert app.extensions["jinjalume"] is extension
assert '<button type="submit"' in rendered
assert 'aria-invalid="true"' in rendered
assert 'id="details"' in rendered
print(rendered[:800].strip())
print("\nSmoke check passed.")

## Component contract map

| Component | Current public macro | Current behavior | Contract risk / decision needed |
| --- | --- | --- | --- |
| Button | `button(label, variant, size, type, href, class_name, disabled)` | Renders `<button>` or `<a>`; three variants and three sizes | Link-disabled behavior and arbitrary HTML attributes are not defined |
| Badge | `badge(label, variant, class_name)` | Renders a styled `<span>`; four variants | No semantic/status role or attribute pass-through contract |
| Alert | `alert(message, variant, title, class_name)` | Renders `role="alert"`; four variants | No live-region strategy, dismiss action, or custom IDs |
| Card | `card(title, description, class_name)` | Requires a caller block and renders a `<section>` | Heading level, landmark labeling, and optional body contract are implicit |
| Input | `input_field(name, label, value, type, placeholder, help_text, error, required, class_name)` | Labeled input with linked help/error text and `aria-invalid` on errors | No extra attrs or explicit `id` escape hatch |
| Select | `select_field(name, label, options, value, help_text, error, required, class_name)` | Native select with selected/disabled options and linked help/error text | No multiple-select or extra attrs contract |
| Textarea | `textarea_field(name, label, value, rows, placeholder, help_text, error, required, class_name)` | Same form-state pattern as input | No extra attrs or explicit `id` escape hatch |
| Avatar | `avatar(name, src, size, class_name)` | Image when `src` exists, otherwise first initial | Initial generation for multi-word names and image failure behavior are undefined |
| Spinner | `spinner(label, size, class_name)` | Accessible status wrapper with visually hidden label | No reduced-motion or visible-label contract |
| Modal | `modal(id, title, description, class_name)` | Native `<dialog>` with linked title/description, a close form, and caller content | No trigger, focus policy, or backdrop interaction contract |

## Missing and partial implementations

These are implementation candidates, not all mandatory changes. The priority assumes the library wants a stable, accessible MVP API before adding a larger component catalog.

| ID | Priority | Area | Evidence in current code | Recommended implementation | Acceptance criteria |
| --- | --- | --- | --- | --- | --- |
| IMP-01 | DONE | Form accessibility | Input, textarea, and select now link help/error messages through deterministic IDs | Retain the relationship contract and add explicit ID/label escape hatches in a future API slice | Screen-reader relationships remain correct for label-only, help-only, error-only, and help+error cases |
| IMP-02 | P0 | Form extensibility | Form macros expose only a fixed small set of HTML attributes | Add a documented attribute strategy (`attrs` mapping or explicit common kwargs) for `autocomplete`, `disabled`, `readonly`, `min`, `max`, `step`, `inputmode`, and `aria-*` | Attributes render safely and predictably without breaking current calls; tests cover booleans and escaped values |
| IMP-03 | P0 | Button semantics | `disabled` is native for `<button>` but only `aria-disabled`/`tabindex` for `<a href=...>` | Define link-button behavior: omit `href`, render a non-link, or require consumer JS; document the choice | Disabled controls cannot be activated through the delivered markup and have a tested accessible name/state |
| IMP-04 | P1 | Extension lifecycle | Every `init_app` call prepends a new `PackageLoader`; repeated initialization can duplicate loaders | Make registration idempotent or document one-time initialization; preserve existing `ChoiceLoader` members | Repeated `init_app` leaves a stable loader chain and `app.extensions` points to the expected instance |
| IMP-05 | P1 | DOM IDs | Form `name` is reused directly as `id` and in error IDs | Accept an explicit ID and define normalization/collision behavior for names such as `user[email]` | Generated IDs are valid, stable, unique within a rendered form, and referenced IDs match |
| IMP-06 | P1 partial | Modal behavior | Modal now links `aria-labelledby` and, when present, `aria-describedby`; no trigger or focus policy is provided | Document native-dialog responsibilities; optionally provide a trigger macro/attribute API | Modal markup has complete relationships; open/close behavior is covered by browser-level or markup tests |
| IMP-07 | P1 | Contract coverage | Tests cover extension loading and a single representative render, not variants, fallback paths, escaping, or package installation | Add parametrized macro contract tests and a built-artifact import/template test | Every public macro has default, non-default, invalid/fallback, accessibility, and escaping coverage |
| IMP-08 | P2 | Component API consistency | All macros hand-roll variant/size dictionaries and accept `class_name`, but not common attributes | Establish naming and composition conventions for variants, sizes, custom classes, IDs, and attributes | New components follow a documented contract and existing components are aligned or explicitly exempted |
| IMP-09 | DONE | Documentation/demo | `docs/components.md` and the local gallery document and render the current macros | Keep docs synchronized as the public API evolves | A contributor can discover every macro signature and see its accessible/error/disabled states locally |
| IMP-10 | DONE | Styling/theme | Semantic tokens, `data-theme=dark`, a demo toggle, and visual snapshots now cover the first theme slice | Add RTL and broader token customization only when there is a concrete consumer contract | Consumers can opt into dark mode without a JavaScript framework or component fork |
| IMP-11 | P3 | Integrations | README roadmap mentions WTForms, HTMX, and Alpine.js, but no adapters exist | Implement integrations as separate optional modules with no new core dependency | Core install remains lightweight; each integration has isolated docs and tests |

In [ ]:
# Keep the backlog executable so it can become issue text or a release plan.
backlog = [
    ("IMP-01", "P0", "Form accessibility relationships", "input.html, textarea.html"),
    ("IMP-02", "P0", "Common HTML attribute pass-through", "input.html, textarea.html"),
    ("IMP-03", "P0", "Disabled link-button semantics", "button.html"),
    ("IMP-04", "P1", "Idempotent extension initialization", "extension.py"),
    ("IMP-05", "P1", "Explicit and safe form IDs", "input.html, textarea.html"),
    ("IMP-06", "P1", "Modal description and native-dialog contract", "modal.html"),
    ("IMP-07", "P1", "Macro contract test matrix", "tests/"),
    ("IMP-08", "P2", "Consistent component API conventions", "all components"),
    ("IMP-09", "P2", "Local component gallery", "demo/ and README.md"),
    ("IMP-10", "P2", "Theme, dark-mode, and RTL policy", "templates and CSS"),
    ("IMP-11", "P3", "Optional WTForms/HTMX/Alpine integrations", "new optional modules"),
]

for issue_id, priority, title, area in backlog:
    print(f"{priority}  {issue_id}  {title}  [{area}]")

## Recommended implementation order

1. Preserve the completed select, theme, docs, HTMX, and visual-regression contracts with regression tests.
2. Add the common HTML attribute strategy and explicit form ID/label escape hatches (`IMP-02` and `IMP-05`) while keeping old positional calls valid.
3. Harden extension lifecycle behavior and define safe disabled-link semantics (`IMP-03` and `IMP-04`).
4. Expand the macro test matrix and package-install checks (`IMP-07`).
5. Only then choose the RTL and optional WTForms/HTMX/Alpine API shape; those decisions affect every future component.

### Definition of done for the next MVP slice

- Existing documented calls render unchanged.
- Every new public argument is documented with one valid example and one edge case.
- Output is tested for semantics, escaping, and fallback behavior.
- `make test`, `make lint`, and `make css` pass.
- The wheel and source distribution contain all required templates.
- The demo visibly exercises default, error, disabled, and dialog states.
- Accessibility decisions are stated in the README instead of being inferred from class names.

In [ ]:
# Optional handoff helper: print issue-ready stubs for the first implementation slice.
issue_details = {}
issue_details["IMP-01"] = (
    "Form field accessibility relationships",
    "Add linked help/error IDs and test all four field-state combinations for "
    "input_field and textarea_field.",
)
issue_details["IMP-02"] = (
    "Common form attribute pass-through",
    "Define and implement the attrs strategy for common HTML and aria attributes "
    "without breaking current macro calls.",
)
issue_details["IMP-03"] = (
    "Disabled button/link semantics",
    "Choose a safe disabled-link representation, implement it, and cover "
    "keyboard and accessible-state expectations in tests.",
)

for issue_id, (title, task) in issue_details.items():
    output = (
        f"## {issue_id}: {title}\n\n### Task\n{task}\n\n"
        "### Validation\n- Add regression tests\n- Run make test\n- Run make lint\n"
    )
    print(output)

## Baseline commands used for this snapshot

Run from the repository root:

```bash
.venv/bin/python -m pytest -q
.venv/bin/ruff check .
npm run build:css
.venv/bin/python -m build --wheel --sdist
```

Expected snapshot: `3 passed`, Ruff clean, Tailwind build complete, and both distributions built successfully.